### Available objects to be displayed in the virtual Space

By running code  ... This text will be filled out by BIBA 








### Code to run

In [1]:
from IPython.display import display, Markdown, IFrame
from ipywidgets import Button, Layout, GridBox
from pathlib import Path
import rospy
from sidecar import Sidecar

from helper import Launcher, Robots, Rvizweb

In [2]:
RVIZWEB_URL = url = rospy.get_param('/rvizweb/jupyter_proxy_url')
IFRAME = IFrame(src=RVIZWEB_URL, width='100%', height='100%')
def MAKE_SIDECAR():
    return Sidecar(title='RVIZWEB', anchor='right')
RVIZWEB = Rvizweb(make_sidecar=MAKE_SIDECAR, iframe=IFRAME, display=display)
ROBOTS = Robots()
LAUNCHER = Launcher(Path.home() / 'work/launch/robots.launch')

In [3]:
def create_button(name):
    btn = Button(
        description=name,
        layout=Layout(width='auto', height='50px'),
        style={'font_size':'1rem'},
        tooltip=f"Launch robot: {name}"
    )
    def launch_robot(name):
        urdf_file = ROBOTS.get_urdf_file(name)
        if urdf_file:
            print(f'Launch robot {name} with urdf file {str(urdf_file)}')
            LAUNCHER.launch(urdf_file=urdf_file,
                            process_name=name,
                            rvizweb=RVIZWEB)
        else:
            print(f'No urdf file for robot "{name}" available')
    btn.on_click(lambda b: launch_robot(name))
    return btn

### Stuff to register

In [4]:
ROBOTS.register('Workstations', Path.home() / 'work/meshes/Workstations/workstations.urdf','Collection of all available benches, desks Tables, etc. Pls. note conveyers etc are static models')
ROBOTS.register('Fences', Path.home() / 'work/meshes/Fences/Fences.urdf','Collection of all available separators, walls, dividers etc.')
ROBOTS.register('Workpieces', Path.home() /  'work/meshes/Workpieces/workpieces.urdf','Collection of all available objects to be placed, picked, machined, or handled, such as tubes, boxes etc.')

In [5]:
 if LAUNCHER.get_process_name() != None:
    display(Markdown(f'### Robot **{LAUNCHER.get_process_name()}** Launched'))
display(Markdown('### Click the buttons below to launch a collection of equipment.'))
display(ROBOTS.get_selection_widget(create_button))

### Click the buttons below to launch a collection of equipment.

GridBox(children=(Button(description='Workstations', layout=Layout(height='50px', width='auto'), style=ButtonS…

###
### Start Colorpicker

In [6]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# Farb-Picker erstellen
color_picker = widgets.ColorPicker(
    value='#ff0000',
    description='Choose color:',
    disabled=False
)

# Text-Input Widget (Farbenname)
text_input = widgets.Text(
    value='myColorName',
    placeholder='Gib Farbnamen ein',
    description='Colorname:',
    disabled=False
)


# Hex → RGB Funktion
def hex_to_rgb(hex_color):
    hex_color = hex_color.lstrip('#')
    r, g, b = (int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    return r, g, b  # RGB einzeln zurückgeben

# Output Widget für Markdown
out = widgets.Output()

# Event-Handler
def on_color_change(change):
    global urdfcolorInput  # <-- make it accessible in other cells
    r, g, b = hex_to_rgb(change['new'])  # in 3 Werte splitten
    r_norm, g_norm, b_norm = r/255, g/255, b/255  # Normiert auf 0-1
    name = text_input.value.strip() or "(kein Name)"
    urdfcolor = f"**URDF-Snippet:** <br> \<material name=\"{name}\" \> <br> &nbsp; &nbsp; \<color rgba =\"{r_norm:.4f} {g_norm:.4f} {b_norm:.4f} 1\"/\> <br> \</material\>"
    urdfcolorInput = f" \<material name=\"{name}\" \> <br> &nbsp; &nbsp; \<color rgba =\"{r_norm:.4f} {g_norm:.4f} {b_norm:.4f} 1\"/\> <br> \</material\>"

    with out:
        clear_output(wait=True)
        display(Markdown(f"**Gewählte Farbe:**\n- R: {r}\n- G: {g}\n- B: {b}")) 
        display(Markdown(urdfcolor))  
   
# Event verbinden
text_input.observe(on_color_change, names='value')
color_picker.observe(on_color_change, names='value')

# Anzeigen
display(text_input,color_picker, out)

Text(value='myColorName', description='Colorname:', placeholder='Gib Farbnamen ein')

ColorPicker(value='#ff0000', description='Choose color:')

Output()

###
### Start Rotation Calculation 

In [7]:
 import math

axes = ["r (rotate x)", "p (rotate y)", "y (rotate z)"]

# Input widgets
deg = [widgets.FloatText(value=0.0, layout=widgets.Layout(width='100px')) for _ in axes]

# Individual radian outputs
rad = [widgets.HTML(f"<div style='text-align:center;'>0.0000</div>") for _ in axes]

# Combined output below table
rad_all = widgets.HTML("<div style='text-align:left;'></div>")

# Update function
def update(change=None):
    radians_values = [math.radians(d.value) for d in deg]
    
    # Update combined output
    rad_all.value = f"<div style='text-align:left;'><br>rpy=\"{' '.join(f'{v:.4f}' for v in radians_values)}\" <br>&nbsp;</div>"
    
    # Update individual radian columns
    for d, r in zip(deg, rad):
        r.value = f"<div style='text-align:center;'>{math.radians(d.value):.4f}</div>"

# Link updates
for d in deg:
    d.observe(update, names='value')

# Table header
header = [widgets.HTML("<b></b>")] + [
    widgets.HTML(f"<div style='text-align:center;'><b>{a}</b></div>", layout=widgets.Layout(width='120px'))
    for a in axes
]

# Degrees row
row_deg = [widgets.HTML("<b>Degrees (°)</b>")] + [
    widgets.HBox([d], layout=widgets.Layout(justify_content='center', width='120px')) for d in deg
]

# Radians row
row_rad = [widgets.HTML("<b>Radians</b>")] + rad

# Grid layout
grid = widgets.GridBox(
    children=header + row_deg + row_rad,
    layout=widgets.Layout(
        grid_template_columns="150px 120px 120px 120px",
        grid_column_gap="8px",
        grid_row_gap="8px"
    )
)

# Display table + combined output
display(grid, rad_all) 

# Initial update
update()

GridBox(children=(HTML(value='<b></b>'), HTML(value="<div style='text-align:center;'><b>r (rotate x)</b></div>…

HTML(value="<div style='text-align:left;'></div>")

In [13]:
  

# Create the text input widget
text_widget = widgets.Text(
    value='',
    placeholder='Type something...',
    description='Input (without *.stl)',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)
  
# Create the output widget
output = widgets.Output()

# Function to display the input in the output widget
def on_text_submit(change):
    output.clear_output()  # Clear previous output 
    global STLname 
    # Use input or fallback
    STLname = change['new'].strip()  or "BallBearings_SKF6256_500x80"
   
    with output:
        display(Markdown(f"**You entered:**  {change['new']}"))

 
# Link the function to the text input (on value change)
text_widget.observe(on_text_submit, names='value')

# Display input and output
display(text_widget, output) 

Text(value='', description='Input (without *.stl)', layout=Layout(width='50%'), placeholder='Type something...…

Output()

In [26]:
  

# Create the text input widget
text_widget = widgets.Text(
    value='',
    placeholder='Type something...',
    description='Input (without *.stl)',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)
  
# Create the output widget
output = widgets.Output()

if 'urdfcolorInput' not in globals():
        urdfcolorInput = f" \<material name=\"Defaultname\" \> <br> &nbsp; &nbsp; \<color rgba =\"0.929 0.109 0.141 1\"/\> <br> \</material\>"
else:
        myInfo = "<BR><i>Material is based on Colorpicker selection above</i> <BR><BR>"
        display(Markdown(myInfo)) 
        
# Function to display the input in the output widget
def on_text_submit(change):
    output.clear_output()  # Clear previous output 
    global STLname 
    # Use input or fallback
    STLname = change['new'].strip()  or "BallBearings_SKF6256_500x80" 

    urdfmesh = f" \<mesh filename=\"../12345/Fences/{STLname}.stl\" /\>  "
    urdforigin = f" &nbsp;&nbsp;&nbsp;&nbsp;&nbsp; \<origin rpy=\"0 0 0\" xyz=\"0 0 0\" /\> "
    urdfgeometry = f" &nbsp;&nbsp;&nbsp;&nbsp;&nbsp; \<geometry><br>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;{urdfmesh} <br>&nbsp;&nbsp;&nbsp;&nbsp; \</geometry\>  "

    urdflink = f"**URDF-Snippet:** <br><br> \<link name=\"{STLname}\" \> <br> &nbsp; &nbsp; \<visual\> <br>{urdforigin} <br>{urdfgeometry}<br> &nbsp; &nbsp; {urdfcolorInput} <br> &nbsp; &nbsp;\</visual\>  <br> \</link\>"

    urdfjoint = f"<br>\<joint name=\"{STLname}ToBaseLink\" type=\"fixed\"\> <br> &nbsp; &nbsp; \<parent link=\"base_link\" /\>  <br> &nbsp; &nbsp; \<child link=\"{STLname}\" /\>  <br> &nbsp;&nbsp; &nbsp;\<origin  rpy=\"0 0 0\"  xyz=\"4 0 0\" /\> <br> \</joint\>"
 
   
    with output:
        display(Markdown(f"**You entered:**  {change['new']}"))
        display(Markdown(urdflink)) 
        display(Markdown(urdfjoint)) 


 
 
# Link the function to the text input (on value change)
text_widget.observe(on_text_submit, names='value')

# Display input and output
display(text_widget, output) 




if 'STLname' not in globals():
    STLname =  "BallBearings_SKF6256_500x80"
else:
    myInfo = "<BR><i>Name used from input Field<BR>"
    display(Markdown(myInfo)) 
 





<BR><i>Material is based on Colorpicker selection above</i> <BR><BR>

Text(value='', description='Input (without *.stl)', layout=Layout(width='50%'), placeholder='Type something...…

Output()

<BR><i>Name used from input Field<BR>